In [183]:
#@title The MIT License (MIT)
#
# Copyright (c) 2026 Eric dos Santos.
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in
# all copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
# THE SOFTWARE.

# Fake News Classification

This project aims to develop a neural network for detecting fake news in Portuguese, using the dataset [Fake.br-Corpus](https://github.com/roneysco/Fake.br-Corpus). With this, we seek to create a system capable of identifying patterns and distinguishing fake news from real news, contributing to the fight against misinformation.

<table class="tfo-notebook-buttons" align="center">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/ericshantos/br_fake_news_detector/blob/main/br_fake_news_detector_model.ipynb
"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run on Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/ericshantos/br_fake_news_detector/blob/main/br_fake_news_detector_model.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View the code on GitHub</a>
  </td>
</table>

## Dataset loading

In [184]:
from abc import ABC, abstractmethod
from typing import Optional
from enum import Enum
from pathlib import Path
import pandas as pd
import os

In [185]:
!git clone https://github.com/roneysco/Fake.br-Corpus
fb_repo = Path("./Fake.br-Corpus/full_texts")

!git clone https://github.com/jpchav98/FakeTrue.Br.git
ft_repo = Path("./FakeTrue.Br")

!git clone https://github.com/Gabriel-Lino-Garcia/FakeRecogna.git
fr_repo = Path("./FakeRecogna/dataset")

fatal: destination path 'Fake.br-Corpus' already exists and is not an empty directory.
fatal: destination path 'FakeTrue.Br' already exists and is not an empty directory.
fatal: destination path 'FakeRecogna' already exists and is not an empty directory.


In [186]:
class BaseExtractor(ABC):
  def __init__(self, path: Path) -> None:
    self.path = path

  @abstractmethod
  def extract(self, label: int) -> pd.DataFrame:
    raise NotImplementedError

In [187]:
class Labels(Enum):
  FAKE = 0
  TRUE = 1

  @classmethod
  def to_dict(cls) -> dict[str, int]:
    return {
      "fake": cls.FAKE.value,
      "true": cls.TRUE.value
    }

## Fake.br Extractor

In [188]:
class FakeBrExtractor(BaseExtractor):
  def __init__(self, path: Path) -> None:
    super().__init__(path)

  def extract(self, labels: int) -> pd.DataFrame:
    news = []

    for dir in os.listdir(self.path):
      if dir not in labels:
        continue

      label = labels[dir]

      for file_path in Path(self.path / dir).glob("*.txt"):
        try:
          content = file_path.read_text(encoding='utf-8')
          news.append({"content": content, "label": label})
        except Exception as e:
          print(f"Error reading file {file_path}: {e}")

    return pd.DataFrame(news)

## FakeTrue.Br Extractor

In [189]:
class FakeTrueExtractor(BaseExtractor):
  def __init__(self, path: Path) -> None:
    super().__init__(path)

  def extract(self, file: str, label: dict[str, int]) -> pd.DataFrame:
    df_raw = pd.read_csv(self.path / file)

    fake_df = pd.DataFrame()
    fake_df['content'] = df_raw['fake']
    fake_df['label'] = label['fake']

    real_df = pd.DataFrame()
    real_df['content'] = df_raw['true']
    real_df['label'] = label['true']

    final_df = pd.concat([fake_df, real_df], ignore_index=True)

    final_df = final_df.dropna(subset=['content'])

    return final_df

## FakeRecogna Extractor

In [190]:
class FakeRecognaExtractor(BaseExtractor):
  def __init__(self, path: Path) -> None:
    super().__init__(path)

  def extract(self, file: str, label: dict[str, int]) -> pd.DataFrame:
    df_raw = pd.read_excel(self.path / file)

    news = df_raw[['Noticia', 'Classe']]

    news = news.rename(columns={
        'Noticia': 'content',
        'Classe': 'label'
    })

    news.dropna(subset=['content', 'label'], inplace=True)

    news['content'] = news['content'].astype(str)
    news['label'] = news['label'].astype(int)

    return news

### News content extraction:


In [191]:
fb_news = FakeBrExtractor(fb_repo).extract(Labels.to_dict())

In [192]:
ft_news = FakeTrueExtractor(ft_repo).extract(
  "FakeTrueBr_corpus.csv", Labels.to_dict()
)

In [193]:
fr_news = FakeRecognaExtractor(fr_repo).extract(
  "FakeRecogna.xlsx", Labels.to_dict()
)

## Data preprocessing

In [194]:
from torch.utils.data import Dataset, DataLoader

### Concatenate the DataFrames

Group Dataframes to generate a single robust database.

In [195]:
data_news = pd.concat(
  [fb_news, ft_news, fr_news], ignore_index=True
).sample(frac=1, random_state=13)

Final base information:

In [196]:
data_news.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22684 entries, 10328 to 338
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   content  22684 non-null  object
 1   label    22684 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 531.7+ KB


## Training

In [197]:
from transformers import AutoTokenizer
import torch

### Splitting the dataset into training and testing

In [198]:
from sklearn.model_selection import train_test_split

In [199]:
X_train, X_test, y_train, y_test = train_test_split(
  data_news['content'],
  data_news['label'],
  test_size=0.1,
  random_state=13,
  stratify=data_news['label']
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Training set size: (20415,)
Test set size: (2269,)


### Create tokenizer

In [200]:
from transformers import AutoTokenizer

In [201]:
tokenizer = AutoTokenizer.from_pretrained(
  "neuralmind/bert-base-portuguese-cased"
)

### Save Tokenizer

In [202]:
tokenizer.save_pretrained("veritas-bert-ptbr")

('veritas-bert-ptbr/tokenizer_config.json', 'veritas-bert-ptbr/tokenizer.json')

## Architectures

In [203]:
class NewsDataset(Dataset):
  def __init__(self, texts, label, tokenizer, max_length=256):
    self.texts = texts.tolist() if hasattr(texts, 'tolist') else texts
    self.label = label.tolist() if hasattr(label, 'tolist') else label
    self.tokenizer = tokenizer

    self.max_length = max_length

  def __getitem__(self, idx: int):
    encoding = self.tokenizer(
        self.texts[idx],
        padding='max_length',
        truncation=True,
        max_length=self.max_length,
        return_tensors="pt"
    )

    return {
        "input_ids": encoding["input_ids"].squeeze(0),
        "attention_mask": encoding["attention_mask"].squeeze(0),
        "label": torch.tensor(self.label[idx], dtype=torch.long)
    }

  def __len__(self) -> int:
    return len(self.texts)

In [204]:
dataset_train = NewsDataset(X_train, y_train, tokenizer)
dataset_test = NewsDataset(X_test, y_test, tokenizer)

In [205]:
loader_train = DataLoader(
  dataset_train,
  batch_size=32,
  shuffle=True,
  pin_memory=True,
  num_workers=2,
  prefetch_factor=2
)

loader_test = DataLoader(
  dataset_test,
  batch_size=32,
  pin_memory=True,
  num_workers=2
)

### Model architecture

In [206]:
import torch.nn as nn
from transformers import AutoModel, PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput
import sys

In [207]:
class BERTClassifierConfig(PretrainedConfig):
  model_type = "bert_classifier"

  def __init__(
    self,
    drop_rate: float = 0.2,
    num_labels: int = 2,
    **kwargs
  ) -> None:
    super().__init__(**kwargs)

    self.drop_rate = drop_rate
    self.num_labels = num_labels

In [208]:
class BERTClassifier(PreTrainedModel):
  config_class = BERTClassifierConfig

  def __init__(self, config: PretrainedConfig) -> None:
    super().__init__(config)

    self.bert = AutoModel.from_pretrained(
      "neuralmind/bert-base-portuguese-cased"
    )

    for param in self.bert.encoder.layer[:6].parameters():
        param.requires_grad = False

    hidden = self.bert.config.hidden_size

    self.classifier = nn.Sequential(
        nn.Linear(hidden, 32),
        nn.GELU(),
        nn.Dropout(config.drop_rate),

        nn.Linear(32, 16),
        nn.GELU(),

        nn.Linear(16, config.num_labels)
    )

    self.post_init()

  def forward(
    self,
    input_ids: Optional[torch.Tensor] = None,
    attention_mask: Optional[torch.Tensor] = None,
    labels: Optional[torch.Tensor] = None
  ) -> SequenceClassifierOutput:
    outputs = self.bert(
      input_ids=input_ids,
      attention_mask=attention_mask
    )

    pooler = outputs.last_hidden_state[:, 0, :]
    logits = self.classifier(pooler)

    loss = None
    if labels is not None:
      loss_fn = nn.CrossEntropyLoss()
      loss = loss_fn(logits, labels)

    return SequenceClassifierOutput(
      loss=loss,
      logits=logits
    )

In [209]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [210]:
config = BERTClassifierConfig()

In [211]:
model = BERTClassifier(config).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Model compilation**:

In [212]:
import torch.optim as optim

In [213]:
optimizer = optim.Adam(model.parameters(), lr=5e-5)

### Training the model

In [214]:
from tqdm import tqdm
from torch.amp import autocast, GradScaler

In [215]:
epochs = 5
scaler = GradScaler("cuda")

In [216]:
accumulation_steps = 4

for epoch in range(epochs):
  model.train()
  total_loss = 0
  optimizer.zero_grad()

  progress_bar = tqdm(loader_train, desc=f"Epoch {epoch+1}")

  for batch_idx, batch in enumerate(progress_bar):

    with autocast("cuda"):
      outputs = model(
          input_ids=batch["input_ids"].to(device),
          attention_mask=batch["attention_mask"].to(device),
          labels=batch["label"].to(device)
      )
      loss = outputs.loss
      loss /= accumulation_steps

    scaler.scale(loss).backward()

    total_loss += loss.item() * accumulation_steps

    if (batch_idx + 1) % accumulation_steps == 0:
      scaler.step(optimizer)
      scaler.update()
      optimizer.zero_grad()

    progress_bar.set_postfix({
      "batch_loss": f"{loss.item() * accumulation_steps:.4f}"
    })

    if batch_idx % 50 == 0:
      torch.cuda.empty_cache()

    del outputs

  avg_loss = total_loss / len(loader_train)
  print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")

Epoch 1: 100%|██████████| 638/638 [03:13<00:00,  3.30it/s, batch_loss=0.4984]


Epoch 1 | Avg Loss: 0.6182


Epoch 2: 100%|██████████| 638/638 [03:13<00:00,  3.30it/s, batch_loss=0.1342]


Epoch 2 | Avg Loss: 0.2944


Epoch 3: 100%|██████████| 638/638 [03:13<00:00,  3.30it/s, batch_loss=0.0994]


Epoch 3 | Avg Loss: 0.1041


Epoch 4: 100%|██████████| 638/638 [03:13<00:00,  3.30it/s, batch_loss=0.0213]


Epoch 4 | Avg Loss: 0.0522


Epoch 5: 100%|██████████| 638/638 [03:12<00:00,  3.32it/s, batch_loss=0.0080]

Epoch 5 | Avg Loss: 0.0285


## Save to model

In [217]:
model.save_pretrained("veritas-bert-ptbr")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Model evaluation

In [226]:
y_true = []
y_pred = []
y_prob = []

model.eval()

with torch.no_grad():
  for batch in loader_test:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)


    logits = model(input_ids, attention_mask).logits

    probs = torch.softmax(logits, dim=1)
    preds = (probs[:, 1] > 0.7).long()

    y_true.extend(labels.cpu().numpy())
    y_pred.extend(preds.cpu().numpy())
    y_prob.extend(probs[:, 1].cpu().numpy())

In [223]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

In [227]:
print("Acurácia:", accuracy_score(y_true, y_pred))
print("Precisão:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

print("\nRelatório completo:")
print(classification_report(y_true, y_pred))

print("\nMatriz de confusão:")
print(confusion_matrix(y_true, y_pred))

print("\nROC-AUC:")
print(roc_auc_score(y_true, y_prob))

Acurácia: 0.9757602468047598
Precisão: 0.9945004582951421
Recall: 0.9567901234567902
F1: 0.9752808988764045

Relatório completo:
              precision    recall  f1-score   support

           0       0.96      0.99      0.98      1135
           1       0.99      0.96      0.98      1134

    accuracy                           0.98      2269
   macro avg       0.98      0.98      0.98      2269
weighted avg       0.98      0.98      0.98      2269


Matriz de confusão:
[[1129    6]
 [  49 1085]]

ROC-AUC:
0.9981656294431624
